# H1. 자치구별 상권 유형에 따른 점포 순감소 차이 분석

### 분석 가설
동일 서비스업종이라도 자치구별 상권 유형에 따라 다음 분기 점포 순감소 여부에 차이가 있을 것이다.

### 분석 기간
2023년 1분기 ~ 2026년 1분기

In [123]:
import pandas as pd

## 0. 라이브러리 불러오기
데이터 불러오기 및 전처리를 위해 pandas를 사용한다.

In [124]:
import os

os.listdir()

['.git',
 'H1_분석.ipynb',
 '서울상권_프로젝트A_심화설계.md',
 '서울시 상권분석서비스(영역-상권).csv',
 '서울시 상권분석서비스(점포-상권).csv',
 '서울시 상권분석서비스(점포-상권)_2024년.csv',
 '서울시_상권분석서비스(점포-상권)_2023년.csv']

In [125]:
df_2023 = pd.read_csv(
    "서울시_상권분석서비스(점포-상권)_2023년.csv",
    encoding="cp949"
)

df_2023.head()

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수
0,20231,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,10,11,9,1,0,0,1
1,20231,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0,0,0
2,20231,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,3,0,0,0,0,0
3,20231,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,0,0,0,1
4,20231,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,1,1,0,0,0,0,0


In [126]:
df_2024 = pd.read_csv(
    "서울시 상권분석서비스(점포-상권)_2024년.csv",
    encoding="cp949"
)

df_2024.head()

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수
0,20241,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,10,11,0,0,0,0,1
1,20241,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0,0,0
2,20241,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,2,2,0,0,50,1,0
3,20241,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,4,5,20,1,0,0,1
4,20241,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,1,1,0,0,0,0,0


In [127]:
df_recent = pd.read_csv(
    "서울시 상권분석서비스(점포-상권).csv",
    encoding="cp949"
)

df_recent.head()

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수
0,20261,R,전통시장,3130327,"평화시장(남평화시장, 제일평화시장, 신평화패션타운)",CS300043,전자상거래업,3,3,0,0,0,0,0
1,20261,R,전통시장,3130327,"평화시장(남평화시장, 제일평화시장, 신평화패션타운)",CS300035,인테리어,2,2,0,0,0,0,0
2,20261,R,전통시장,3130327,"평화시장(남평화시장, 제일평화시장, 신평화패션타운)",CS300029,애완동물,1,1,0,0,0,0,0
3,20261,R,전통시장,3130327,"평화시장(남평화시장, 제일평화시장, 신평화패션타운)",CS300027,섬유제품,1,1,0,0,0,100,1
4,20261,R,전통시장,3130327,"평화시장(남평화시장, 제일평화시장, 신평화패션타운)",CS300024,운동/경기용품,5,5,0,0,0,0,0


## 1. 점포-상권 데이터 불러오기 및 통합

서울시 상권분석서비스의 2023년, 2024년, 2025~2026년 점포-상권 데이터를 불러왔다.

연도별 파일의 컬럼 구성을 확인한 결과, 점포 관련 컬럼명이 변경되어 있어 동일한 의미의 변수를 기준으로 컬럼명을 통일하였다.

- 2023~2024년 `점포_수` ↔ 최근 데이터 `일반_점포_수`
- 2023~2024년 `유사_업종_점포_수` ↔ 최근 데이터 `전체_점포_수`

분기별 점포 수를 동일한 기준으로 비교하기 위해 최근 데이터의 `일반_점포_수`를 `점포_수`로, `전체_점포_수`를 `유사_업종_점포_수`로 변경하였다.

이후 세 데이터를 `concat`하여 2023년 1분기부터 2026년 1분기까지 하나의 데이터프레임으로 통합하였다.

In [128]:
print(df_2023.columns.tolist())
print(df_2024.columns.tolist())
print(df_recent.columns.tolist())

['기준_년분기_코드', '상권_구분_코드', '상권_구분_코드_명', '상권_코드', '상권_코드_명', '서비스_업종_코드', '서비스_업종_코드_명', '점포_수', '유사_업종_점포_수', '개업_율', '개업_점포_수', '폐업_률', '폐업_점포_수', '프랜차이즈_점포_수']
['기준_년분기_코드', '상권_구분_코드', '상권_구분_코드_명', '상권_코드', '상권_코드_명', '서비스_업종_코드', '서비스_업종_코드_명', '점포_수', '유사_업종_점포_수', '개업_율', '개업_점포_수', '폐업_률', '폐업_점포_수', '프랜차이즈_점포_수']
['기준_년분기_코드', '상권_구분_코드', '상권_구분_코드_명', '상권_코드', '상권_코드_명', '서비스_업종_코드', '서비스_업종_코드_명', '전체_점포_수', '일반_점포_수', '프랜차이즈_점포_수', '개업_율', '개업_점포_수', '폐업_률', '폐업_점포_수']


In [129]:
df_recent = df_recent.rename(columns={
    "일반_점포_수": "점포_수",
    "전체_점포_수": "유사_업종_점포_수"
})

In [130]:
df_all = pd.concat(
    [df_2023, df_2024, df_recent],
    ignore_index=True
)

df_all.head()

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수
0,20231,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,10,11,9,1,0,0,1
1,20231,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0,0,0
2,20231,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,3,0,0,0,0,0
3,20231,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,0,0,0,1
4,20231,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,1,1,0,0,0,0,0


### 데이터 기간 확인

통합 데이터에 포함된 분기를 확인하여 분석 기간이 연속적으로 구성되어 있는지 점검한다.

In [131]:
print(df_all.shape)
print(df_all["기준_년분기_코드"].unique())

(995377, 14)
[20231 20232 20233 20234 20241 20242 20243 20244 20261 20254 20253 20252
 20251]


In [132]:
sorted(df_all["기준_년분기_코드"].unique())

[np.int64(20231),
 np.int64(20232),
 np.int64(20233),
 np.int64(20234),
 np.int64(20241),
 np.int64(20242),
 np.int64(20243),
 np.int64(20244),
 np.int64(20251),
 np.int64(20252),
 np.int64(20253),
 np.int64(20254),
 np.int64(20261)]

- 분석 기간: 2023년 1분기 ~ 2026년 1분기
- 총 13개 분기
- 통합 데이터 크기: 995,377행 × 15열
- 분석 기간 내 분기 누락 없음

## 2. 통합 데이터 구조 확인

통합된 데이터의 기본 구조와 중복 여부를 확인한다.

분기별·상권별·서비스업종별 점포 수를 비교해야 하므로,
`기준_년분기_코드 + 상권_코드 + 서비스_업종_코드` 조합이 하나의 관측치를 구분할 수 있는지 확인한다.

In [133]:
df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 995377 entries, 0 to 995376
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   기준_년분기_코드    995377 non-null  int64
 1   상권_구분_코드     995377 non-null  str  
 2   상권_구분_코드_명   995377 non-null  str  
 3   상권_코드        995377 non-null  int64
 4   상권_코드_명      995377 non-null  str  
 5   서비스_업종_코드    995377 non-null  str  
 6   서비스_업종_코드_명  995377 non-null  str  
 7   점포_수         995377 non-null  int64
 8   유사_업종_점포_수   995377 non-null  int64
 9   개업_율         995377 non-null  int64
 10  개업_점포_수      995377 non-null  int64
 11  폐업_률         995377 non-null  int64
 12  폐업_점포_수      995377 non-null  int64
 13  프랜차이즈_점포_수   995377 non-null  int64
dtypes: int64(9), str(5)
memory usage: 106.3 MB


In [134]:
key_cols = [
    "기준_년분기_코드",
    "상권_코드",
    "서비스_업종_코드"
]

df_all.duplicated(subset=key_cols).sum()

np.int64(0)

### 데이터 구조 확인 결과

- 전체 데이터: 995,377행 × 15열
- 분석 기간: 2023년 1분기 ~ 2026년 1분기 (총 13개 분기)
- `기준_년분기_코드 + 상권_코드 + 서비스_업종_코드` 기준 중복: 0건
- 순감소 여부 계산에 필요한 `점포_수`에는 결측치가 없음


### 데이터 구조 확인 결과

- 전체 데이터: 995,377행 × 15열
- 분석 기간: 2023년 1분기 ~ 2026년 1분기 (총 13개 분기)
- `기준_년분기_코드 + 상권_코드 + 서비스_업종_코드` 기준 중복: 0건
- 순감소 여부 계산에 필요한 `점포_수`에는 결측치가 없음
- `유사_업종_점포_수`, `일반_점포_수`는 연도별 데이터 구조 차이로 결측치가 발생했으며, 현재 H1 분석에는 사용하지 않음

In [135]:
df_area = pd.read_csv(
    "서울시 상권분석서비스(영역-상권).csv",
    encoding="cp949"
)

df_area.head()

,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,엑스좌표_값,와이좌표_값,자치구_코드,자치구_코드_명,행정동_코드,행정동_코드_명,영역_면적
0,A,골목상권,3110055,황학동벼룩시장,201642,452260,11140,중구,11140670,황학동,27575
1,A,골목상권,3110008,배화여자대학교(박노수미술관),197093,453418,11110,종로구,11110515,청운효자동,149264
2,A,골목상권,3110009,자하문터널,196991,455057,11110,종로구,11110550,부암동,178306
3,A,골목상권,3110010,평창동서측,197064,456643,11110,종로구,11110560,평창동,369415
4,A,골목상권,3110017,정독도서관,198581,453781,11110,종로구,11110600,가회동,83855


In [136]:
df_area.columns.tolist()


['상권_구분_코드',
 '상권_구분_코드_명',
 '상권_코드',
 '상권_코드_명',
 '엑스좌표_값',
 '와이좌표_값',
 '자치구_코드',
 '자치구_코드_명',
 '행정동_코드',
 '행정동_코드_명',
 '영역_면적']

In [137]:
area_key = df_area[
    ["상권_코드", "자치구_코드", "자치구_코드_명"]
].drop_duplicates()

area_key.head()

,상권_코드,자치구_코드,자치구_코드_명
0,3110055,11140,중구
1,3110008,11110,종로구
2,3110009,11110,종로구
3,3110010,11110,종로구
4,3110017,11110,종로구


In [138]:
area_key["상권_코드"].duplicated().sum()

np.int64(0)

### 자치구 정보 결합

`상권_코드`가 영역-상권 데이터에서 중복되지 않는 것을 확인한 뒤,
점포-상권 데이터에 자치구 정보를 결합하였다.

In [139]:
df_all = df_all.merge(
    area_key,
    on="상권_코드",
    how="left"
)

df_all.head()

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수,자치구_코드,자치구_코드_명
0,20231,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,10,11,9,1,0,0,1,11110,종로구
1,20231,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0,0,0,11110,종로구
2,20231,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,3,0,0,0,0,0,11110,종로구
3,20231,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,0,0,0,1,11110,종로구
4,20231,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,1,1,0,0,0,0,0,11110,종로구


In [140]:
print(df_all.shape)
print(df_all["자치구_코드_명"].isna().sum())

(995377, 16)
0


In [141]:
df_h1 = df_all[
    [
        "기준_년분기_코드",
        "자치구_코드_명",
        "상권_구분_코드_명",
        "상권_코드",
        "상권_코드_명",
        "서비스_업종_코드",
        "서비스_업종_코드_명",
        "점포_수",
        "개업_율",
        "개업_점포_수",
        "폐업_률",
        "폐업_점포_수"
    ]
].copy()

df_h1.head()

,기준_년분기_코드,자치구_코드_명,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수
0,20231,종로구,골목상권,3110001,이북5도청사,CS100001,한식음식점,10,9,1,0,0
1,20231,종로구,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,0,0,0,0
2,20231,종로구,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,0,0,0,0
3,20231,종로구,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,0,0,0,0
4,20231,종로구,골목상권,3110001,이북5도청사,CS100010,커피-음료,1,0,0,0,0


## 4. H1 분석용 변수 정리

H1 분석에 필요한 분기, 자치구, 상권유형, 서비스업종, 점포 수를 중심으로 변수를 정리한다.

In [142]:
df_h1 = df_all[
    [
        "기준_년분기_코드",
        "자치구_코드_명",
        "상권_구분_코드_명",
        "상권_코드",
        "상권_코드_명",
        "서비스_업종_코드",
        "서비스_업종_코드_명",
        "점포_수",
        "개업_율",
        "개업_점포_수",
        "폐업_률",
        "폐업_점포_수"
    ]
].copy()

df_h1.head()

,기준_년분기_코드,자치구_코드_명,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수
0,20231,종로구,골목상권,3110001,이북5도청사,CS100001,한식음식점,10,9,1,0,0
1,20231,종로구,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,0,0,0,0
2,20231,종로구,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,0,0,0,0
3,20231,종로구,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,0,0,0,0
4,20231,종로구,골목상권,3110001,이북5도청사,CS100010,커피-음료,1,0,0,0,0


### 상권 유형 및 서비스업종 확인

분석에 앞서 상권 유형과 서비스업종이 어떤 범주로 구성되어 있는지 확인한다.

In [143]:
df_h1["상권_구분_코드_명"].value_counts()

상권_구분_코드_명
골목상권    616014
발달상권    233207
전통시장    139405
관광특구      6751
Name: count, dtype: int64

In [144]:
print("서비스업종 수:", df_h1["서비스_업종_코드_명"].nunique())

서비스업종 수: 100


In [145]:
df_h1["서비스_업종_코드_명"].value_counts()

서비스_업종_코드_명
한식음식점     20474
부동산중개업    20114
커피-음료     19757
일반의류      19321
미용실       19179
          ...  
한복점        1638
볼링장        1151
DVD방       1076
고시원        1031
중고가구        910
Name: count, Length: 100, dtype: int64

## 5. 다음 분기 점포 순감소 여부 생성

동일한 상권·서비스업종의 점포 수를 분기 순서대로 비교하기 위해
상권과 서비스업종별로 데이터를 묶고 분기 순으로 정렬한다.

이후 다음 분기의 점포 수가 현재 분기보다 감소한 경우를 `1`,
감소하지 않은 경우를 `0`으로 정의한다.

In [146]:
df_h1 = df_h1.sort_values(
    ["상권_코드", "서비스_업종_코드", "기준_년분기_코드"]
).reset_index(drop=True)

df_h1.head(15)

,기준_년분기_코드,자치구_코드_명,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수
0,20231,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,117,4,5,4,5
1,20232,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,119,2,3,1,1
2,20233,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,117,3,4,4,5
3,20234,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,118,2,2,1,1
4,20241,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,122,8,10,3,4
5,20242,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,126,9,12,6,8
6,20243,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,128,8,11,4,6
7,20244,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,132,5,7,3,4
8,20251,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,135,5,8,3,5
9,20252,용산구,관광특구,3001491,이태원 관광특구,CS100001,한식음식점,134,3,5,5,7


### 다음 분기 점포 수 생성

같은 상권·같은 서비스업종 안에서 바로 다음 분기의 점포 수를 가져와,
현재 분기 점포 수와 비교할 수 있도록 한다.

In [147]:
df_h1["다음_분기_점포_수"] = (
    df_h1
    .groupby(["상권_코드", "서비스_업종_코드"])["점포_수"]
    .shift(-1)
)

df_h1[
    [
        "기준_년분기_코드",
        "상권_코드",
        "서비스_업종_코드_명",
        "점포_수",
        "다음_분기_점포_수"
    ]
].head(15)

,기준_년분기_코드,상권_코드,서비스_업종_코드_명,점포_수,다음_분기_점포_수
0,20231,3001491,한식음식점,117,119.0
1,20232,3001491,한식음식점,119,117.0
2,20233,3001491,한식음식점,117,118.0
3,20234,3001491,한식음식점,118,122.0
4,20241,3001491,한식음식점,122,126.0
5,20242,3001491,한식음식점,126,128.0
6,20243,3001491,한식음식점,128,132.0
7,20244,3001491,한식음식점,132,135.0
8,20251,3001491,한식음식점,135,134.0
9,20252,3001491,한식음식점,134,137.0


In [148]:
df_h1["다음_분기_코드"] = (
    df_h1
    .groupby(["상권_코드", "서비스_업종_코드"])["기준_년분기_코드"]
    .shift(-1)
)

df_h1[
    [
        "기준_년분기_코드",
        "다음_분기_코드",
        "상권_코드",
        "서비스_업종_코드_명"
    ]
].head(20)

,기준_년분기_코드,다음_분기_코드,상권_코드,서비스_업종_코드_명
0,20231,20232.0,3001491,한식음식점
1,20232,20233.0,3001491,한식음식점
2,20233,20234.0,3001491,한식음식점
3,20234,20241.0,3001491,한식음식점
4,20241,20242.0,3001491,한식음식점
5,20242,20243.0,3001491,한식음식점
6,20243,20244.0,3001491,한식음식점
7,20244,20251.0,3001491,한식음식점
8,20251,20252.0,3001491,한식음식점
9,20252,20253.0,3001491,한식음식점


### 다음 분기 순감소 여부 생성

현재 분기의 점포 수와 다음 분기의 점포 수를 비교하여,
다음 분기 점포 수가 감소한 경우 `1`, 감소하지 않은 경우 `0`으로 정의한다.

단, 다음 분기 데이터가 없는 경우에는 순감소 여부를 판단할 수 없으므로 결측값으로 처리한다.

In [149]:
import numpy as np

df_h1["다음_분기_순감소"] = np.where(
    df_h1["다음_분기_점포_수"].isna(),
    np.nan,
    (df_h1["다음_분기_점포_수"] < df_h1["점포_수"]).astype(int)
)

In [150]:
quarter_order = [
    20231, 20232, 20233, 20234,
    20241, 20242, 20243, 20244,
    20251, 20252, 20253, 20254,
    20261
]

next_quarter = {
    quarter_order[i]: quarter_order[i + 1]
    for i in range(len(quarter_order) - 1)
}

df_h1["정상_다음_분기"] = df_h1["기준_년분기_코드"].map(next_quarter)

wrong_gap = (
    df_h1["다음_분기_코드"].notna()
    & (df_h1["다음_분기_코드"] != df_h1["정상_다음_분기"])
)


print("분기를 건너뛴 관측치:", wrong_gap.sum())

분기를 건너뛴 관측치: 566


In [151]:
# 연속된 다음 분기만 남기기
df_h1.loc[wrong_gap, "다음_분기_순감소"] = np.nan

print("분기 건너뛰기 제외:", wrong_gap.sum())
print("\n다음_분기_순감소 분포")
print(df_h1["다음_분기_순감소"].value_counts(dropna=False))

print(
    "\n분석 가능 데이터:",
    df_h1["다음_분기_순감소"].notna().sum()
)

분기 건너뛰기 제외: 566

다음_분기_순감소 분포
다음_분기_순감소
0.0    842509
NaN     82578
1.0     70290
Name: count, dtype: int64

분석 가능 데이터: 912799


In [152]:
df_h1[
    [
        "기준_년분기_코드",
        "상권_코드_명",
        "서비스_업종_코드_명",
        "점포_수",
        "다음_분기_점포_수",
        "다음_분기_순감소"
    ]
].head(20)

,기준_년분기_코드,상권_코드_명,서비스_업종_코드_명,점포_수,다음_분기_점포_수,다음_분기_순감소
0,20231,이태원 관광특구,한식음식점,117,119.0,0.0
1,20232,이태원 관광특구,한식음식점,119,117.0,1.0
2,20233,이태원 관광특구,한식음식점,117,118.0,0.0
3,20234,이태원 관광특구,한식음식점,118,122.0,0.0
4,20241,이태원 관광특구,한식음식점,122,126.0,0.0
5,20242,이태원 관광특구,한식음식점,126,128.0,0.0
6,20243,이태원 관광특구,한식음식점,128,132.0,0.0
7,20244,이태원 관광특구,한식음식점,132,135.0,0.0
8,20251,이태원 관광특구,한식음식점,135,134.0,1.0
9,20252,이태원 관광특구,한식음식점,134,137.0,0.0


In [153]:
df_h1["다음_분기_순감소"].value_counts(dropna=False)

다음_분기_순감소
0.0    842509
NaN     82578
1.0     70290
Name: count, dtype: int64

In [154]:
df_h1.loc[
    df_h1["다음_분기_순감소"].isna(),
    "기준_년분기_코드"
].value_counts().sort_index()

기준_년분기_코드
20231      588
20232      537
20233      533
20234      477
20241      651
20242      596
20243      559
20244      467
20251      603
20252      539
20253      594
20254      462
20261    75972
Name: count, dtype: int64

In [155]:
(df_h1["점포_수"] == 0).value_counts()

점포_수
False    960553
True      34824
Name: count, dtype: int64

In [156]:
df_h1[
    (df_h1["다음_분기_순감소"].isna()) &
    (df_h1["기준_년분기_코드"] != 20261)
][
    [
        "기준_년분기_코드",
        "자치구_코드_명",
        "상권_코드",
        "상권_코드_명",
        "서비스_업종_코드",
        "서비스_업종_코드_명",
        "점포_수"
    ]
].head(20)

,기준_년분기_코드,자치구_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수
231,20253,용산구,3001491,이태원 관광특구,CS200009,동물병원,0
1755,20244,중구,3001492,명동 남대문 북창동 다동 무교동 관광특구,CS200046,의류임대,0
2228,20241,중구,3001492,명동 남대문 북창동 다동 무교동 관광특구,CS300039,모터사이클및부품,0
3000,20253,중구,3001493,동대문패션타운 관광특구,CS300005,주류도매,0
3384,20243,중구,3001493,동대문패션타운 관광특구,CS300037,중고차판매,0
3733,20253,종로구,3001494,종로?청계 관광특구,CS200014,회계사사무소,0
5080,20244,송파구,3001495,잠실 관광특구,CS200026,자동차미용,0
6159,20241,강남구,3001496,강남 마이스 관광특구,CS200026,자동차미용,0
6397,20243,강남구,3001496,강남 마이스 관광특구,CS300008,수산물판매,0
6676,20242,강남구,3001496,강남 마이스 관광특구,CS300033,철물점,0


In [157]:
df_h1.loc[
    (df_h1["다음_분기_순감소"].isna()) &
    (df_h1["기준_년분기_코드"] != 20261),
    "점포_수"
].value_counts()

점포_수
0    6541
1      64
2       1
Name: count, dtype: int64

In [158]:
weird = df_h1[
    (df_h1["다음_분기_순감소"].isna()) &
    (df_h1["기준_년분기_코드"] != 20261) &
    (df_h1["점포_수"] > 0)
]

weird[
    [
        "기준_년분기_코드",
        "자치구_코드_명",
        "상권_코드",
        "상권_코드_명",
        "서비스_업종_코드",
        "서비스_업종_코드_명",
        "점포_수"
    ]
].head(20)

,기준_년분기_코드,자치구_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수
24425,20254,중구,3110043,충무초등학교,CS200002,외국어학원,1
71245,20243,성동구,3110130,답십리역 6번,CS200038,독서실,1
94955,20244,광진구,3110165,국립정신건강센터,CS300036,조명용품,1
97130,20252,광진구,3110168,대원고등학교,CS300033,철물점,1
106178,20251,동대문구,3110184,용두사거리,CS300029,애완동물,1
108699,20252,동대문구,3110189,제기동역 1번,CS200043,건축물청소,1
113576,20254,동대문구,3110199,홍릉시장,CS300026,완구,1
124947,20254,동대문구,3110220,외대앞역 1번,CS300029,애완동물,1
126323,20254,동대문구,3110225,촬영소사거리,CS100005,제과점,1
136166,20253,중랑구,3110241,먹골역 5번,CS200013,기타법무서비스,1


### 다음 분기 관측 불가 데이터 처리

다음 분기 점포 수를 확인할 수 없는 관측치는 순감소 여부를 판단할 수 없으므로 분석 대상에서 제외한다.

대부분은 현재 점포 수가 0인 관측치였으며, 일부 점포가 존재하는 경우도 확인되었으나 데이터만으로 폐업·누락 등의 원인을 구분할 수 없어 임의로 순감소 여부를 부여하지 않았다.

## 6. 최종 분석 데이터 구성

다음 분기 점포 수가 확인되는 관측치만 사용하여 H1 분석용 데이터를 구성한다.

다음 분기 정보가 없는 경우에는 점포 순감소 여부를 판단할 수 없으므로 분석 대상에서 제외한다.

In [159]:
df_model = df_h1[
    df_h1["다음_분기_순감소"].notna()
].copy()

print("분석 데이터 수:", len(df_model))

df_model["다음_분기_순감소"].value_counts()

분석 데이터 수: 912799


다음_분기_순감소
0.0    842509
1.0     70290
Name: count, dtype: int64

In [160]:
df_model["다음_분기_순감소"].value_counts(normalize=True) * 100

다음_분기_순감소
0.0    92.29951
1.0     7.70049
Name: proportion, dtype: float64

## 7. 상권 유형별 점포 순감소 비율 확인

상권 유형에 따라 다음 분기 점포 순감소 비율에 차이가 있는지 확인한다.

먼저 골목상권, 발달상권, 전통시장, 관광특구별 순감소 비율을 비교한다.

In [161]:
decrease_by_type = (
    df_model
    .groupby("상권_구분_코드_명")["다음_분기_순감소"]
    .agg(["count", "sum", "mean"])
)

decrease_by_type["순감소율(%)"] = decrease_by_type["mean"] * 100

decrease_by_type

,count,sum,mean,순감소율(%)
상권_구분_코드_명,,,,
골목상권,564379,35494.0,0.062890,6.289036
관광특구,6222,1184.0,0.190293,19.029251
발달상권,214505,25223.0,0.117587,11.758700
전통시장,127693,8389.0,0.065697,6.569663


### 1차 확인 결과

상권 유형별 단순 순감소율을 비교한 결과,
관광특구(19.03%)와 발달상권(11.75%)에서 상대적으로 높은 순감소율이 나타났으며,
전통시장(6.56%)과 골목상권(6.28%)은 상대적으로 낮게 나타났다.

다만 이는 업종과 자치구의 차이를 고려하지 않은 단순 비교이므로,
상권 유형 자체의 차이라고 해석할 수는 없다.
이후 동일 업종 및 자치구 조건을 고려하여 차이를 추가로 확인한다.

In [162]:
df_model = df_h1[
    df_h1["다음_분기_순감소"].notna()
].copy()

print("분석 데이터 수:", len(df_model))

df_model["다음_분기_순감소"].value_counts()

분석 데이터 수: 912799


다음_분기_순감소
0.0    842509
1.0     70290
Name: count, dtype: int64